<a href="https://colab.research.google.com/github/Rogerio-mack/Modelos-de-Linguagem-e-Generativos-2026S1/blob/main/3_1_Language_Models_and_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.1 Language Models and Attention

## 1. INTRODUCTION

In this notebook, we will explore different forms of attention mechanisms in sequence modeling.
We'll start with a simple RNN model without attention, then move to an LSTM-based sequence-to-sequence
model that uses a "classic" attention mechanism. Finally, we'll look at a minimal example of self-attention,
but we won't dive into the full Transformer architecture.

Attention helps models:
- Focus on relevant parts of an input sequence.
- Mitigate the issues of long sequences in standard RNNs/LSTMs.
- Learn to align and weight input elements dynamically when predicting outputs.


In [1]:
# 2. SIMPLE RNN MODEL

# We'll build a small synthetic dataset to demonstrate a simple classification task,
# and then define a basic RNN model (using PyTorch in this example) to classify sequences.
# We'll compare the results later when we add attention mechanisms.

import sys
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Generate a Synthetic Dataset

We create a simple sequence classification dataset. Each sequence is a random collection of
integers, and the label is determined by some simple pattern (e.g., sum of the sequence mod 2).

In [2]:
def generate_synthetic_data(num_samples=1000, seq_length=10, vocab_size=50):
    """
    Generate synthetic data.
    Each sequence is a random list of integers in [1, vocab_size],
    and the label is (sum of sequence) mod 2 as a simple classification target.
    """
    X = np.random.randint(1, vocab_size, size=(num_samples, seq_length))
    y = np.sum(X, axis=1) % 2  # binary classification
    return X, y

num_samples = 5000
seq_length = 10
vocab_size = 50

X_data, y_data = generate_synthetic_data(num_samples, seq_length, vocab_size)

# Split into train/test
split_idx = int(num_samples * 0.8)
X_train, X_test = X_data[:split_idx], X_data[split_idx:]
y_train, y_test = y_data[:split_idx], y_data[split_idx:]

# Convert to PyTorch tensors
X_train_t = torch.LongTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_test_t = torch.LongTensor(X_test).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

In [12]:
X_train

array([[39, 29, 15, ..., 19, 23, 11],
       [11, 24, 36, ...,  2, 24, 44],
       [30, 38,  2, ..., 44, 25, 49],
       ...,
       [27, 29,  7, ..., 12, 39, 27],
       [45, 33, 15, ..., 49, 15, 20],
       [19,  1,  4, ...,  5,  9,  1]])

## 2. Define a Simple RNN Model for Classification

The model will:
1. Embed the input tokens.
2. Pass them through an RNN.
3. Take the final hidden state for classification.

In [3]:
class SimpleRNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super(SimpleRNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)           # (batch, seq, embed_dim)
        output, hidden = self.rnn(embedded)    # output: (batch, seq, hidden_dim), hidden: (1, batch, hidden_dim)
        # For classification, we'll use the final hidden state
        final_hidden = hidden.squeeze(0)       # (batch, hidden_dim)
        logits = self.fc(final_hidden)         # (batch, num_classes)
        return logits

# Instantiate the model
embed_dim = 16
hidden_dim = 32
num_classes = 2

simple_rnn_model = SimpleRNNModel(vocab_size, embed_dim, hidden_dim, num_classes).to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(simple_rnn_model.parameters(), lr=0.0003)

### Training Loop for the Simple RNN

We'll do a basic training loop over a fixed number of epochs.

In [4]:
def train_model(model, criterion, optimizer, X_train, y_train, epochs=10, batch_size=32):
    model.train()
    num_samples = X_train.size(0)
    for epoch in range(epochs):
        permutation = torch.randperm(num_samples)
        epoch_loss = 0.0
        correct = 0

        for i in range(0, num_samples, batch_size):
            optimizer.zero_grad()

            indices = permutation[i:i+batch_size]
            batch_x, batch_y = X_train[indices], y_train[indices]

            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == batch_y).sum().item()

        avg_loss = epoch_loss / (num_samples // batch_size)
        accuracy = correct / num_samples
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Accuracy: {accuracy*100:.2f}%")

train_model(simple_rnn_model, criterion, optimizer, X_train_t, y_train_t, epochs=10, batch_size=32)

Epoch 1/10, Loss: 0.7046, Accuracy: 48.02%
Epoch 2/10, Loss: 0.6953, Accuracy: 49.98%
Epoch 3/10, Loss: 0.6920, Accuracy: 51.32%
Epoch 4/10, Loss: 0.6904, Accuracy: 52.62%
Epoch 5/10, Loss: 0.6890, Accuracy: 53.57%
Epoch 6/10, Loss: 0.6880, Accuracy: 54.25%
Epoch 7/10, Loss: 0.6869, Accuracy: 54.60%
Epoch 8/10, Loss: 0.6860, Accuracy: 54.45%
Epoch 9/10, Loss: 0.6854, Accuracy: 55.07%
Epoch 10/10, Loss: 0.6845, Accuracy: 55.30%


### Evaluate Simple RNN on Test Set


In [5]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        preds = torch.argmax(logits, dim=1)
        accuracy = (preds == y_test).float().mean().item()
    return accuracy

test_accuracy = evaluate_model(simple_rnn_model, X_test_t, y_test_t)
print(f"Test Accuracy of Simple RNN: {test_accuracy*100:.2f}%")

Test Accuracy of Simple RNN: 49.10%


## 3. LSTM with ATTENTION

Now we'll build an LSTM-based sequence model that includes a simple attention mechanism.
The attention mechanism will allow the model to learn a weighted combination of LSTM hidden states
across timesteps, rather than just relying on the final hidden state.

Steps:
1. Embed the input tokens.
2. Pass them through an LSTM.
3. Compute attention weights over hidden states.
4. Use weighted sum of hidden states as context vector.
5. Pass context vector to a classification layer.

Neste exemplo olhando para um mecanismo de **Atenção Global** (muitas vezes chamado de **Atenção de Bahdanau simplificada**). O objetivo aqui é resolver o "gargalo" das RNNs: em vez de forçar a rede a compactar toda a informação de uma frase no último estado oculto ($h_n$), permitimos que ela "olhe de volta" para todos os estados da sequência.



# Como o código funciona

O segredo está na transformação do lstm_out em um único context_vector. Vamos decompor a lógica:

1. A Saída da LSTM (`lstm_out`):

  Ao contrário de uma LSTM comum onde pegaríamos apenas o último hidden state, aqui mantemos todos os estados da sequência. Se sua frase tem 10 palavras, você tem 10 vetores representando cada momento da leitura.

2. Cálculo dos Scores (`attention_scores`):

  `self.attention = nn.Linear(hidden_dim, 1, bias=False)`
  
  Aqui, cada um dos 10 vetores passa por uma camada linear que reduz sua dimensão para 1. Pense nisso como a rede atribuindo uma "nota de importância" bruta para cada palavra da frase.

3. Normalização (`softmax`):

  O Softmax transforma essas notas brutas em probabilidades (pesos que somam 1).Se a palavra 3 for muito relevante para a classificação, o `attention_weights[3]` será próximo de 0.9, enquanto os outros serão próximos de 0.

4. Vetor de Contexto (`context_vector`):

Multiplicamos cada vetor original da LSTM pelo seu respectivo peso e somamos tudo.$$\text{context_vector} = \sum_{i=1}^{seq} \text{attention_weight}_i \times \text{lstm_out}_i$$O resultado é um resumo inteligente da frase, onde as partes importantes foram "amplificadas" e as irrelevantes "silenciadas".

# Auto-Atenção

No código anterior a atenção é estática em relação à própria sequência. Cada palavra recebe um peso baseado apenas nela mesma (através da camada linear).

Na Auto-Atenção (**Self-Attention**) usada nos Transformers, a dinâmica muda: A importância de uma palavra depende das outras palavras da frase.

`Queries, Keys e Values`: Em vez de uma camada linear simples, usamos três projeções. A palavra "banco" terá um peso diferente se a palavra "rio" ou "dinheiro" estiver na mesma sequência.

| Característica | Atenção Global (Bahdanau) | Auto-Atenção (Transformer) |
|-|-|-|
| Dependência|Cada estado é avaliado individualmente.|Cada estado interage com todos os outros.|
|Cálculo|Camada Linear sobre o estado oculto.|Produto escalar entre Query e Key|
|Propósito|Criar um resumo global da sequência.|Criar representações ricas contextuais de cada palavra.|

In [6]:
class LSTMAttentionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super(LSTMAttentionModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.attention = nn.Linear(hidden_dim, 1, bias=False)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)               # (batch, seq, embed_dim)
        lstm_out, (h, c) = self.lstm(embedded)     # lstm_out: (batch, seq, hidden_dim)

        # Compute attention scores
        # attention_scores: (batch, seq, 1)
        attention_scores = self.attention(lstm_out)

        # Attention weights = softmax over seq dimension
        attention_weights = torch.softmax(attention_scores, dim=1)   # (batch, seq, 1)

        # Weighted sum of lstm_out
        # (batch, seq, hidden_dim) * (batch, seq, 1) -> (batch, seq, hidden_dim) -> sum over seq
        context_vector = torch.sum(lstm_out * attention_weights, dim=1)  # (batch, hidden_dim)

        logits = self.fc(context_vector)         # (batch, num_classes)
        return logits

lstm_att_model = LSTMAttentionModel(vocab_size, embed_dim, hidden_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_att_model.parameters(), lr=0.0003)

### Train the LSTM with Attention
We reuse our training function.

In [7]:
train_model(lstm_att_model, criterion, optimizer, X_train_t, y_train_t, epochs=10, batch_size=32)

Epoch 1/10, Loss: 0.6937, Accuracy: 50.65%
Epoch 2/10, Loss: 0.6933, Accuracy: 51.10%
Epoch 3/10, Loss: 0.6932, Accuracy: 51.60%
Epoch 4/10, Loss: 0.6928, Accuracy: 51.80%
Epoch 5/10, Loss: 0.6925, Accuracy: 52.15%
Epoch 6/10, Loss: 0.6923, Accuracy: 51.25%
Epoch 7/10, Loss: 0.6921, Accuracy: 51.65%
Epoch 8/10, Loss: 0.6920, Accuracy: 51.65%
Epoch 9/10, Loss: 0.6914, Accuracy: 52.20%
Epoch 10/10, Loss: 0.6912, Accuracy: 52.55%


### Evaluate the LSTM with Attention

In [8]:
test_accuracy = evaluate_model(lstm_att_model, X_test_t, y_test_t)
print(f"Test Accuracy of LSTM + Attention: {test_accuracy*100:.2f}%")

Test Accuracy of LSTM + Attention: 50.80%


## 4. INTRODUCTION TO SELF-ATTENTION

Finally, let's explore a minimal example of self-attention (not a full Transformer). We'll define
a simple "SelfAttention" layer that computes pairwise attention across timesteps. We'll then
integrate it into a small classifier. This is only a demonstration, so it won't be heavily optimized.

In [9]:
class SelfAttention(nn.Module):
    """
    A minimal self-attention layer:
    - Projects inputs into queries, keys, values
    - Computes attention scores = Q*K^T / sqrt(d_k)
    - Weighted sum of values
    """
    def __init__(self, hidden_dim):
        super(SelfAttention, self).__init__()
        self.hidden_dim = hidden_dim
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        """
        x shape: (batch_size, seq_length, hidden_dim)
        returns: (batch_size, seq_length, hidden_dim)
        """
        Q = self.query_proj(x)   # (batch, seq, hidden_dim)
        K = self.key_proj(x)     # (batch, seq, hidden_dim)
        V = self.value_proj(x)   # (batch, seq, hidden_dim)

        # Compute attention scores
        # scores: (batch, seq, seq)
        scores = torch.bmm(Q, K.transpose(1, 2)) / (self.hidden_dim ** 0.5)

        # weights: (batch, seq, seq)
        weights = torch.softmax(scores, dim=-1)

        # out: (batch, seq, hidden_dim)
        out = torch.bmm(weights, V)
        return out

class SelfAttentionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super(SelfAttentionModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.self_attention = SelfAttention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)               # (batch, seq, embed_dim)
        lstm_out, (h, c) = self.lstm(embedded)     # (batch, seq, hidden_dim)

        # Apply self-attention over the LSTM outputs
        sa_out = self.self_attention(lstm_out)     # (batch, seq, hidden_dim)

        # We can pool (e.g. average) the self-attention output across seq dimension
        context_vector = torch.mean(sa_out, dim=1) # (batch, hidden_dim)

        logits = self.fc(context_vector)           # (batch, num_classes)
        return logits

self_att_model = SelfAttentionModel(vocab_size, embed_dim, hidden_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(self_att_model.parameters(), lr=0.0003)

### Train the Self-Attention Model


In [10]:
train_model(self_att_model, criterion, optimizer, X_train_t, y_train_t, epochs=10, batch_size=32)

Epoch 1/10, Loss: 0.6934, Accuracy: 51.45%
Epoch 2/10, Loss: 0.6928, Accuracy: 51.62%
Epoch 3/10, Loss: 0.6924, Accuracy: 51.30%
Epoch 4/10, Loss: 0.6921, Accuracy: 51.82%
Epoch 5/10, Loss: 0.6918, Accuracy: 51.68%
Epoch 6/10, Loss: 0.6914, Accuracy: 52.85%
Epoch 7/10, Loss: 0.6910, Accuracy: 52.68%
Epoch 8/10, Loss: 0.6904, Accuracy: 53.50%
Epoch 9/10, Loss: 0.6901, Accuracy: 53.33%
Epoch 10/10, Loss: 0.6894, Accuracy: 53.65%


### Evaluate the Self-Attention Model

In [11]:
test_accuracy = evaluate_model(self_att_model, X_test_t, y_test_t)
print(f"Test Accuracy of LSTM + Self-Attention: {test_accuracy*100:.2f}%")

Test Accuracy of LSTM + Self-Attention: 49.00%
